<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/TDConnect4Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitbully[gui]

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import re
import zipfile

import numpy as np

# If you want a static type check against the Protocol:
from bitbully.agent_interface import Connect4Agent
from bitbully import Board


# =============================================================================
# Models
# =============================================================================

@dataclass(frozen=True, slots=True)
class TupleLUT:
    """One n-tuple LUT (keeps both original + mirrored index sets)."""
    n: int
    m: int
    idxs: np.ndarray        # (n,), original indices (mirror1)
    idxs_m: np.ndarray      # (n,), mirrored indices (mirror2)
    lut: np.ndarray         # (P**n,), float64


@dataclass(frozen=True, slots=True)
class PlayerWeights:
    """Weights file for one 'player to move' (p0 or p1)."""
    t: int
    p: int
    luts: tuple[TupleLUT, ...]


@dataclass(frozen=True, slots=True)
class TwoPlayerWeights:
    """Weights for both players to move: p0 (yellow), p1 (red)."""
    p0: PlayerWeights
    p1: PlayerWeights

    def for_player(self, player: int) -> PlayerWeights:
        if player == 0:
            return self.p0
        if player == 1:
            return self.p1
        raise ValueError("player must be 0 (yellow-to-move) or 1 (red-to-move)")


# =============================================================================
# Parsing helpers (text -> TwoPlayerWeights)
# =============================================================================

@dataclass(slots=True)
class Block:
    text: str | list[str]
    children: list["Block"]


_ALLOWED_CHARS = re.compile(r"^[\s0-9+\-\.eE{}]*$")


def _normalize_text(text: str) -> str:
    text = text.replace("\r", "")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _validate_text(text: str) -> None:
    if text.count("{") != text.count("}"):
        raise ValueError("Brace count mismatch: unbalanced '{' and '}'.")
    if not _ALLOWED_CHARS.match(text):
        for i, ch in enumerate(text):
            if not re.match(r"[\s0-9+\-\.eE{}]", ch):
                raise ValueError(f"Invalid character {ch!r} (ord={ord(ch)}) at position {i}")
        raise ValueError("Invalid character(s) found.")


def _parse_blocks(text: str) -> Block:
    root = Block(text="", children=[])
    stack: list[Block] = [root]
    buf: list[str] = []

    def flush() -> None:
        if not buf:
            return
        s = "".join(buf)
        buf.clear()
        cur = stack[-1]
        assert isinstance(cur.text, str)
        cur.text += s

    for pos, ch in enumerate(text):
        if ch == "{":
            flush()
            child = Block(text="", children=[])
            stack[-1].children.append(child)
            stack.append(child)
        elif ch == "}":
            flush()
            stack.pop()
            if not stack:
                raise ValueError(f"Unmatched '}}' at character index {pos}")
        else:
            buf.append(ch)

    if len(stack) != 1:
        raise ValueError("Unmatched '{' (file ended while blocks were still open).")

    flush()
    return root


def _split_tokens_inplace(block: Block) -> None:
    if isinstance(block.text, str):
        s = block.text.strip()
        block.text = s.split() if s else []
    for child in block.children:
        _split_tokens_inplace(child)


def _block_to_tuple_lut(block: Block, *, p: int, int_dtype=np.int32) -> TupleLUT:
    if not isinstance(block.text, list) or len(block.text) != 2:
        raise ValueError(f"Expected tuple header tokens [N, M], got: {block.text!r}")
    if len(block.children) != 3:
        raise ValueError(f"Expected 3 children (set, mirror, weights), got {len(block.children)}")

    n = int(block.text[0])
    m = int(block.text[1])

    b_set, b_mirror, b_w = block.children
    if not isinstance(b_set.text, list) or not isinstance(b_mirror.text, list) or not isinstance(b_w.text, list):
        raise ValueError("Child blocks must be token lists. Did you call _split_tokens_inplace()?")

    if len(b_set.text) != n:
        raise ValueError(f"Sample set length mismatch: expected {n}, got {len(b_set.text)}")
    if len(b_mirror.text) != n:
        raise ValueError(f"Mirrored set length mismatch: expected {n}, got {len(b_mirror.text)}")

    expected_w = int(p**n)
    if len(b_w.text) != expected_w:
        raise ValueError(f"LUT length mismatch: expected {expected_w} (= {p}^{n}), got {len(b_w.text)}")

    return TupleLUT(
        n=n,
        m=m,
        idxs=np.asarray([int(x) for x in b_set.text], dtype=int_dtype),
        idxs_m=np.asarray([int(x) for x in b_mirror.text], dtype=int_dtype),
        lut=np.asarray([float(x) for x in b_w.text], dtype=np.float64),
    )


class TDWeightsLoader:
    """Load TD-agent weights from `p0.txt` and `p1.txt` (directory or zip)."""

    def __init__(self, *, int_dtype=np.int32, validate: bool = True, strict_t: bool = True) -> None:
        self._int_dtype = int_dtype
        self._validate = validate
        self._strict_t = strict_t

    def _load_from_text(self, raw: str) -> PlayerWeights:
        text = _normalize_text(raw)
        if self._validate:
            _validate_text(text)

        root = _parse_blocks(text)
        _split_tokens_inplace(root)

        if not root.children:
            raise ValueError("No top-level block found. Expected file to start with '{'.")
        file_block = root.children[0]

        if not isinstance(file_block.text, list) or len(file_block.text) < 2:
            raise ValueError(f"Expected file header tokens [T, P], got: {file_block.text!r}")

        t = int(file_block.text[0])
        p = int(file_block.text[1])

        luts = tuple(_block_to_tuple_lut(b, p=p, int_dtype=self._int_dtype) for b in file_block.children)

        if self._validate and self._strict_t and t != len(luts):
            raise ValueError(f"T mismatch: header T={t}, parsed tuples={len(luts)}")

        return PlayerWeights(t=t, p=p, luts=luts)

    def load_file(self, path: str | Path) -> PlayerWeights:
        raw = Path(path).read_text(encoding="utf-8", errors="strict")
        return self._load_from_text(raw)

    def load_file_from_zip(self, zip_path: str | Path, member: str) -> PlayerWeights:
        with zipfile.ZipFile(zip_path, "r") as zf:
            try:
                data = zf.read(member)
            except KeyError as e:
                raise FileNotFoundError(f"{member!r} not found in zip {str(zip_path)!r}") from e
        raw = data.decode("utf-8", errors="strict")
        return self._load_from_text(raw)

    def load_two_player(self, directory: str | Path, *, p0: str = "p0.txt", p1: str = "p1.txt") -> TwoPlayerWeights:
        d = Path(directory)
        w0 = self.load_file(d / p0)
        w1 = self.load_file(d / p1)
        if self._validate and (w0.p != w1.p):
            raise ValueError(f"P mismatch between p0/p1: {w0.p} vs {w1.p}")
        return TwoPlayerWeights(p0=w0, p1=w1)

    def load_two_player_from_zip(
        self,
        zip_path: str | Path,
        *,
        p0: str = "p0.txt",
        p1: str = "p1.txt",
    ) -> TwoPlayerWeights:
        w0 = self.load_file_from_zip(zip_path, p0)
        w1 = self.load_file_from_zip(zip_path, p1)
        if self._validate and (w0.p != w1.p):
            raise ValueError(f"P mismatch between p0/p1: {w0.p} vs {w1.p}")
        return TwoPlayerWeights(p0=w0, p1=w1)


# =============================================================================
# Board encoding (7 columns x 6 rows, bottom->top) -> flat 42 state array
# =============================================================================

def board_cols_to_flat_features(board_cols: list[list[int]]) -> np.ndarray:
    """Convert board_cols[col][row] into 42-length state array with P=4 encoding."""
    if len(board_cols) != 7 or any(len(col) != 6 for col in board_cols):
        raise ValueError("Expected board_cols shape (7, 6): 7 columns each with 6 cells.")

    flat = np.zeros(42, dtype=np.int8)

    for col in range(7):
        column = board_cols[col]

        reachable_row: int | None = None
        for row in range(6):
            if column[row] == 0:
                reachable_row = row
                break

        for row in range(6):
            idx = col * 6 + row
            cell = column[row]
            if cell == 1:
                flat[idx] = 1
            elif cell == 2:
                flat[idx] = 2
            else:
                flat[idx] = 3 if (reachable_row is not None and row == reachable_row) else 0

    return flat


# =============================================================================
# TD evaluator
# =============================================================================

def _lut_index_from_states(states: np.ndarray, p: int) -> int:
    powers = p ** np.arange(states.size, dtype=np.int64)
    return int(states.astype(np.int64, copy=False) @ powers)


class TDEvaluator:
    """Compute TD n-tuple value from TwoPlayerWeights."""

    def __init__(self, weights: TwoPlayerWeights) -> None:
        self._weights = weights

    def value(self, *, board_cols: list[list[int]], player_to_move: int) -> float:
        pw = self._weights.for_player(player_to_move)
        flat = board_cols_to_flat_features(board_cols).astype(np.int64, copy=False)

        total = 0.0
        for tup in pw.luts:
            idx1 = _lut_index_from_states(flat[tup.idxs], pw.p)
            idx2 = _lut_index_from_states(flat[tup.idxs_m], pw.p)
            total += float(tup.lut[idx1]) + float(tup.lut[idx2])
        return total

    @staticmethod
    def to_score(raw_value: float) -> int:
        return int(round(100.0 * math.tanh(raw_value)))


# =============================================================================
# Agent that satisfies the Connect4Agent Protocol
# =============================================================================

class TDConnect4Agent:
    """TD n-tuple agent compatible with `Connect4Agent` Protocol.

    Implements:
      - score_all_moves(board) -> dict[int,int]
      - best_move(board) -> int
      - score_move(board, column, first_guess=0) -> int
    """

    def __init__(self, evaluator: TDEvaluator, *, tie_break: str = "center") -> None:
        self._eval = evaluator
        self._tie_break = tie_break

    # ---- Protocol method 1 ----
    def score_all_moves(self, board) -> dict[int, int]:
        """Return {col: score} for all legal moves. Illegal/full columns are excluded."""
        player_to_move = board.current_player() - 1  # BitBully: {1,2} -> {0,1}
        if player_to_move not in (0, 1):
            raise ValueError(f"Unexpected current_player(): {board.current_player()}")

        scores: dict[int, int] = {}

        for col in range(7):
            if not board.is_legal_move(col):
                continue

            after = board.play_on_copy(col)
            next_player = after.current_player() - 1
            if next_player not in (0, 1):
                raise ValueError(f"Unexpected current_player() after move: {after.current_player()}")

            raw = self._eval.value(board_cols=after.to_array(), player_to_move=next_player)
            score = self._eval.to_score(raw)

            # normalize to "bigger is better for side-to-move"
            if player_to_move == 1:
                score = -score

            scores[col] = score

        return scores

    # ---- Protocol method 2 ----
    def best_move(self, board) -> int:
        """Return best legal move using BitBully-like tie breaking."""
        scores = self.score_all_moves(board)
        if not scores:
            raise ValueError("No legal moves available.")

        best = max(scores.values())
        candidates = [c for c, v in scores.items() if v == best]

        if len(candidates) == 1:
            return candidates[0]

        if self._tie_break == "center":
            center = 3
            return min(candidates, key=lambda c: (abs(c - center), c))
        if self._tie_break == "left":
            return min(candidates)
        if self._tie_break == "right":
            return max(candidates)

        raise ValueError("tie_break must be one of: 'center', 'left', 'right'")

    # ---- Optional Protocol method ----
    def score_move(self, board, column: int, first_guess: int = 0) -> int:
        """Evaluate a single legal move. `first_guess` is accepted for Protocol compatibility."""
        _ = first_guess  # this TD agent ignores it
        if not board.is_legal_move(column):
            raise ValueError(f"Illegal move: column {column}")

        player_to_move = board.current_player() - 1
        after = board.play_on_copy(column)
        next_player = after.current_player() - 1

        raw = self._eval.value(board_cols=after.to_array(), player_to_move=next_player)
        score = self._eval.to_score(raw)

        if player_to_move == 1:
            score = -score

        return score


# Export to standardized format:

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import io
import json
import zipfile

import numpy as np


# Assumes your dataclasses exist in scope:
# TupleLUT, PlayerWeights, TwoPlayerWeights


_FORMAT_NAME = "bitbully.td_weights.zip"
_FORMAT_VERSION = 1


def _write_npy_to_zip(zf: zipfile.ZipFile, name: str, arr: np.ndarray) -> None:
    """Write a numpy array as a .npy file into the zip."""
    buf = io.BytesIO()
    np.save(buf, np.asarray(arr), allow_pickle=False)
    zf.writestr(name, buf.getvalue())


def _read_npy_from_zip(zf: zipfile.ZipFile, name: str) -> np.ndarray:
    """Read a numpy array from a .npy file inside the zip."""
    raw = zf.read(name)
    with io.BytesIO(raw) as buf:
        arr = np.load(buf, allow_pickle=False)
    if not isinstance(arr, np.ndarray):
        raise TypeError(f"{name!r} did not decode to an ndarray.")
    return arr


def export_two_player_weights_zip(path: str | Path, both: TwoPlayerWeights) -> None:
    """
    Export `both` (TwoPlayerWeights) to a single zip file containing:
      - meta.json
      - arrays as .npy files

    This avoids pickle and is stable across platforms / Python versions.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    meta: dict = {
        "format": _FORMAT_NAME,
        "version": _FORMAT_VERSION,
        "players": {},
    }

    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as zf:

        def write_player(tag: str, pw: PlayerWeights) -> None:
            player_meta: dict = {
                "t": int(pw.t),
                "p": int(pw.p),
                "luts": [],
            }

            for i, tup in enumerate(pw.luts):
                base = f"{tag}/lut_{i:04d}"

                # Store arrays as .npy
                idxs_name = f"{base}/idxs.npy"
                idxs_m_name = f"{base}/idxs_m.npy"
                lut_name = f"{base}/lut.npy"

                _write_npy_to_zip(zf, idxs_name, tup.idxs)
                _write_npy_to_zip(zf, idxs_m_name, tup.idxs_m)
                _write_npy_to_zip(zf, lut_name, tup.lut)

                # Store the small scalar metadata in JSON
                player_meta["luts"].append(
                    {
                        "n": int(tup.n),
                        "m": int(tup.m),
                        "idxs": idxs_name,
                        "idxs_m": idxs_m_name,
                        "lut": lut_name,
                    }
                )

            meta["players"][tag] = player_meta

        write_player("p0", both.p0)
        write_player("p1", both.p1)

        # Write meta.json last
        zf.writestr("meta.json", json.dumps(meta, indent=2, sort_keys=True).encode("utf-8"))


def import_two_player_weights_zip(
    path: str | Path,
    *,
    int_dtype=np.int32,
    validate: bool = True,
    strict_t: bool = True,
) -> TwoPlayerWeights:
    """
    Import TwoPlayerWeights from the zip written by export_two_player_weights_zip().

    Args:
        int_dtype: dtype for idx arrays (e.g. np.int32).
        validate: check basic invariants (p match, lengths).
        strict_t: enforce pw.t equals number of LUTs.
    """
    path = Path(path)

    with zipfile.ZipFile(path, "r") as zf:
        meta_raw = zf.read("meta.json").decode("utf-8")
        meta = json.loads(meta_raw)

        if meta.get("format") != _FORMAT_NAME:
            raise ValueError(f"Unexpected format: {meta.get('format')!r}")
        if int(meta.get("version", -1)) != _FORMAT_VERSION:
            raise ValueError(f"Unexpected version: {meta.get('version')!r}")

        players = meta.get("players")
        if not isinstance(players, dict) or "p0" not in players or "p1" not in players:
            raise ValueError("meta.json missing players p0/p1.")

        def read_player(tag: str) -> PlayerWeights:
            pm = players[tag]
            t = int(pm["t"])
            p = int(pm["p"])
            lut_entries = pm["luts"]
            if not isinstance(lut_entries, list):
                raise ValueError(f"players[{tag!r}].luts must be a list.")

            luts: list[TupleLUT] = []
            for entry in lut_entries:
                n = int(entry["n"])
                m = int(entry["m"])

                idxs = _read_npy_from_zip(zf, entry["idxs"]).astype(int_dtype, copy=False)
                idxs_m = _read_npy_from_zip(zf, entry["idxs_m"]).astype(int_dtype, copy=False)
                lut = _read_npy_from_zip(zf, entry["lut"]).astype(np.float64, copy=False)

                if validate:
                    if idxs.ndim != 1 or idxs.size != n:
                        raise ValueError(f"{tag}: idxs shape mismatch (expected ({n},), got {idxs.shape}).")
                    if idxs_m.ndim != 1 or idxs_m.size != n:
                        raise ValueError(f"{tag}: idxs_m shape mismatch (expected ({n},), got {idxs_m.shape}).")
                    expected = int(p**n)
                    if lut.ndim != 1 or lut.size != expected:
                        raise ValueError(
                            f"{tag}: lut length mismatch (expected {expected} (= {p}^{n}), got {lut.size})."
                        )

                luts.append(TupleLUT(n=n, m=m, idxs=idxs, idxs_m=idxs_m, lut=lut))

            if validate and strict_t and t != len(luts):
                raise ValueError(f"{tag}: T mismatch: header t={t}, tuples={len(luts)}")

            return PlayerWeights(t=t, p=p, luts=tuple(luts))

        p0 = read_player("p0")
        p1 = read_player("p1")

        if validate and p0.p != p1.p:
            raise ValueError(f"P mismatch between p0/p1: {p0.p} vs {p1.p}")

        return TwoPlayerWeights(p0=p0, p1=p1)

In [ ]:
if False:
  loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)
  both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

  # Export to a clean single file:
  export_two_player_weights_zip("td_weights_clean.tdw.zip", both)

# Later / elsewhere:
both2 = import_two_player_weights_zip("td_weights_clean.tdw.zip", int_dtype=np.int32)

evaluator = TDEvaluator(both2)
td_agent = TDConnect4Agent(evaluator, tie_break="center")

In [ ]:
# =============================================================================
# Example usage (including Protocol check)
# =============================================================================
if False:
  loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)

  # Directory:
  # both = loader.load_two_player(".")

  # Zip:
  both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

  evaluator = TDEvaluator(both)
  td_agent = TDConnect4Agent(evaluator, tie_break="center")


import bitbully
board = bitbully.Board()
board.play("3334101133114410")

# Structural/runtime Protocol check (optional):
from bitbully.agent_interface import Connect4Agent
assert isinstance(td_agent, Connect4Agent)

print(td_agent.score_all_moves(board))
print("best:", td_agent.best_move(board))
print("score col 3:", td_agent.score_move(board, 3))


In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent
bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [ ]:
agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

In [ ]:
from __future__ import annotations

from typing import Protocol

def play_match(
    agent_yellow: Connect4Agent,
    agent_red: Connect4Agent,
    *,
    start: bitbully.Board | None = None,
    max_plies: int = 42,
    verbose: int = 0,
) -> int:
    """
    Play a full game between two agents on your BitBully `Board`.

    Returns:
        0 -> draw
        1 -> yellow win
        2 -> red win
    """
    board = start.copy() if start is not None else bitbully.Board()

    # Safety: if someone passes a terminal board
    if board.is_game_over():
        w = board.winner()
        return 0 if w is None else int(w)

    plies = 0
    while not board.is_game_over():
        if plies >= max_plies:
            # Should never happen in Connect-4, but keeps us robust
            return 0

        player = board.current_player()  # 1=yellow, 2=red
        agent = agent_yellow if player == 1 else agent_red

        move = agent.best_move(board)

        if not board.is_legal_move(move):
            raise ValueError(
                f"Agent selected illegal move {move} for player {player}. "
                f"Legal moves: {board.legal_moves()}"
            )

        ok = board.play(move)
        if not ok:
            # `play()` returns False on illegal moves; we already checked, so this would indicate a bug.
            raise RuntimeError(f"Board.play({move}) returned False although move was legal.")

        plies += 1

        if verbose >= 1:
            print(board)

    w = board.winner()
    return 0 if w is None else int(w)

In [ ]:
import bitbully

result = play_match(td_agent, bitbully_agent, verbose=1)
# 0 draw, 1 yellow win, 2 red win
print(result)

In [ ]:
from collections import defaultdict

n_matches = 200
total_scores = defaultdict(int)
for _ in range(n_matches):
    result = play_match(td_agent, bitbully_agent, verbose=0)
    total_scores[result] += 1

In [ ]:
total_scores

In [ ]:
"""BitBullyArena: a lightweight Connect-4 arena for pitting agents against each other.

Design goals (v1):
- Deterministic runs via a global seed and per-game derived seeds (schedule order independent).
- Best-effort timeouts (no subprocess): measure after `best_move()` returns.
- ε-randomization per agent: each agent has its own ε-schedule; when colors swap,
  each agent keeps its ε for that game.
- Illegal move / exception / timeout => immediate loss (logged + recorded).
- Agents receive only a copy of the board (`board.copy()`).
- Optional score logging (`score_all_moves`) for diagnostics.

Assumptions (based on your Board semantics):
- Yellow == Player 1 == integer 1, starts the game.
- Red    == Player 2 == integer 2, goes second.

You can drop this module into your project and adapt naming/paths as needed.
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
import hashlib
import logging
import random
import time

from bitbully import Board
from bitbully.agent_interface import Connect4Agent


# -----------------------------
# Public configuration + types
# -----------------------------

class Color(int, Enum):
    """Player color / identity mapping (matches `Board.current_player()`)."""
    YELLOW = 1  # Player 1, starts
    RED = 2     # Player 2


@dataclass(frozen=True, slots=True)
class AgentSpec:
    agent_id: str
    agent: Connect4Agent
    colors: tuple[Color, ...] = (Color.YELLOW, Color.RED)

    # Per-agent epsilon schedule (sweep). This is NOT tied to color.
    # When colors swap, each agent keeps its epsilon for that game.
    epsilons: tuple[float, ...] = (0.0,)

    def can_play(self, color: Color) -> bool:
        return color in self.colors


@dataclass(frozen=True, slots=True)
class TimeControl:
    """Time controls for agents.

    Best-effort timeouts (no subprocess): we measure elapsed time and forfeit
    *after* `best_move()` returns if a limit is exceeded.
    """
    per_move_timeout_s: float | None = None
    per_game_budget_s: float | None = None


@dataclass(frozen=True, slots=True)
class ArenaConfig:
    """Configuration for running an arena tournament."""
    agents: tuple[AgentSpec, ...]
    n_games: int
    time_control: TimeControl = TimeControl()
    seed: int = 0

    log_scores: bool = False
    use_tqdm: bool = False
    logger: logging.Logger | None = None


class TerminationReason(str, Enum):
    CONNECT4 = "connect4"
    DRAW = "draw"
    ILLEGAL_MOVE = "illegal_move"
    EXCEPTION = "exception"
    TIMEOUT = "timeout"
    INCOMPATIBLE_CONSTRAINTS = "incompatible_constraints"


@dataclass(frozen=True, slots=True)
class GamePlayers:
    """Assignment for a single game."""
    yellow_id: str
    red_id: str


@dataclass(frozen=True, slots=True)
class GameConfig:
    players: GamePlayers
    # Epsilon associated with each agent_id for this game instance.
    # Example: {"bitbully": 0.0, "td": 0.10}
    epsilon_by_agent: dict[str, float]
    seed: int


@dataclass(frozen=True, slots=True)
class MoveMeta:
    ply: int
    player: Color
    agent_id: str
    epsilon: float
    move: int
    was_epsilon_random: bool
    elapsed_s: float
    remaining_budget_s: float | None
    # Optional: store score dictionary for that move decision.
    # Only present if ArenaConfig.log_scores=True and call succeeded.
    scores: dict[int, int] | None = None


@dataclass(frozen=True, slots=True)
class GameRecord:
    """Complete record of a single played game."""
    game_cfg: GameConfig
    moves: tuple[int, ...]
    move_meta: tuple[MoveMeta, ...]
    winner: Color | None
    termination: TerminationReason
    # Extra diagnostic info (exception text, illegal move, etc.)
    detail: str | None = None


@dataclass(frozen=True, slots=True)
class SkippedPairing:
    """Record a pairing that was not scheduled due to incompatible constraints."""
    agent_a: str
    agent_b: str
    reason: TerminationReason = TerminationReason.INCOMPATIBLE_CONSTRAINTS
    detail: str | None = None


@dataclass(frozen=True, slots=True)
class AggregateRow:
    """Aggregate W/L/D counts for a matchup under a specific epsilon and color assignment.

    eps values are the epsilons of the *agents assigned to those colors* in that game.
    """
    agent_yellow: str
    agent_red: str
    epsilon_yellow: float
    epsilon_red: float
    games: int
    yellow_wins: int
    red_wins: int
    draws: int
    timeouts: int
    illegal_moves: int
    exceptions: int


@dataclass(frozen=True, slots=True)
class ArenaResult:
    games: tuple[GameRecord, ...]
    aggregates: tuple[AggregateRow, ...]
    skipped: tuple[SkippedPairing, ...]


# -----------------------------
# Example baseline agent
# -----------------------------

class RandomAgent:
    """Agent that plays a uniformly random legal move."""

    def score_all_moves(self, board: Board) -> dict[int, int]:
        return {c: 0 for c in board.legal_moves()}

    def best_move(self, board: Board) -> int:
        return random.choice(board.legal_moves())


# -----------------------------
# Arena implementation
# -----------------------------

class BitBullyArena:
    def __init__(self) -> None:
        pass

    def run(self, cfg: ArenaConfig) -> ArenaResult:
        logger = cfg.logger or logging.getLogger(__name__)

        # Validate config basics
        if cfg.n_games <= 0:
            raise ValueError(f"n_games must be > 0, got {cfg.n_games}")

        for spec in cfg.agents:
            if len(spec.epsilons) == 0:
                raise ValueError(f"Agent {spec.agent_id}: epsilons must not be empty.")
            for eps in spec.epsilons:
                if not (0.0 <= eps <= 1.0):
                    raise ValueError(f"Agent {spec.agent_id}: epsilon must be in [0,1], got {eps}")

        # Plan schedule:
        # for each unordered pairing (i<j), run both color swaps and all (eps_a, eps_b) combinations.
        agent_specs = list(cfg.agents)
        skipped: list[SkippedPairing] = []
        planned_games: list[GameConfig] = []

        for i in range(len(agent_specs)):
            for j in range(i + 1, len(agent_specs)):
                a = agent_specs[i]
                b = agent_specs[j]

                for eps_a in a.epsilons:
                    for eps_b in b.epsilons:
                        eps_map = {a.agent_id: eps_a, b.agent_id: eps_b}

                        # assignment 1: A yellow, B red
                        if a.can_play(Color.YELLOW) and b.can_play(Color.RED):
                            for k in range(cfg.n_games):
                                planned_games.append(
                                    GameConfig(
                                        players=GamePlayers(yellow_id=a.agent_id, red_id=b.agent_id),
                                        epsilon_by_agent=eps_map,
                                        seed=_derive_game_seed(
                                            cfg.seed,
                                            a.agent_id,
                                            b.agent_id,
                                            Color.YELLOW,
                                            Color.RED,
                                            eps_a,
                                            eps_b,
                                            k,
                                        ),
                                    )
                                )
                        else:
                            skipped.append(
                                SkippedPairing(
                                    agent_a=a.agent_id,
                                    agent_b=b.agent_id,
                                    detail=(
                                        f"Cannot assign {a.agent_id}=YELLOW and {b.agent_id}=RED under constraints."
                                    ),
                                )
                            )

                        # assignment 2: B yellow, A red
                        if b.can_play(Color.YELLOW) and a.can_play(Color.RED):
                            for k in range(cfg.n_games):
                                planned_games.append(
                                    GameConfig(
                                        players=GamePlayers(yellow_id=b.agent_id, red_id=a.agent_id),
                                        epsilon_by_agent=eps_map,
                                        seed=_derive_game_seed(
                                            cfg.seed,
                                            a.agent_id,
                                            b.agent_id,
                                            Color.RED,
                                            Color.YELLOW,
                                            eps_a,
                                            eps_b,
                                            k,
                                        ),
                                    )
                                )
                        else:
                            skipped.append(
                                SkippedPairing(
                                    agent_a=a.agent_id,
                                    agent_b=b.agent_id,
                                    detail=(
                                        f"Cannot assign {b.agent_id}=YELLOW and {a.agent_id}=RED under constraints."
                                    ),
                                )
                            )

        # Map agent_id -> agent instance
        agent_by_id: dict[str, Connect4Agent] = {s.agent_id: s.agent for s in agent_specs}

        # Optional progress bar
        iterator = planned_games
        if cfg.use_tqdm:
            try:
                from tqdm import tqdm  # type: ignore[import-not-found]
                iterator = tqdm(planned_games, desc="BitBullyArena", unit="game")
            except Exception:
                iterator = planned_games

        games_out: list[GameRecord] = []
        for game_cfg in iterator:
            gr = self._play_one_game(
                game_cfg=game_cfg,
                agent_by_id=agent_by_id,
                time_control=cfg.time_control,
                seed=game_cfg.seed,
                log_scores=cfg.log_scores,
                logger=logger,
            )
            games_out.append(gr)

        aggregates = _aggregate_games(games_out)

        return ArenaResult(
            games=tuple(games_out),
            aggregates=tuple(aggregates),
            skipped=tuple(skipped),
        )

    def _play_one_game(
        self,
        *,
        game_cfg: GameConfig,
        agent_by_id: dict[str, Connect4Agent],
        time_control: TimeControl,
        seed: int,
        log_scores: bool,
        logger: logging.Logger,
    ) -> GameRecord:
        rng = random.Random(seed)

        yellow = agent_by_id[game_cfg.players.yellow_id]
        red = agent_by_id[game_cfg.players.red_id]

        board = Board()  # always start empty (no opening noise)
        moves: list[int] = []
        meta: list[MoveMeta] = []

        # Remaining per-game budgets (seconds)
        budget_y: float | None = time_control.per_game_budget_s
        budget_r: float | None = time_control.per_game_budget_s

        while not board.is_game_over():
            player_int = board.current_player()
            player = Color(player_int)

            agent_id = game_cfg.players.yellow_id if player == Color.YELLOW else game_cfg.players.red_id
            agent = yellow if player == Color.YELLOW else red
            eps = game_cfg.epsilon_by_agent[agent_id]

            # Choose move (epsilon-random or agent)
            board_for_agent = board.copy()  # pass a copy to agent
            ply = board.count_tokens()

            start = time.perf_counter()
            was_eps = False
            scores: dict[int, int] | None = None

            try:
                if rng.random() < eps:
                    was_eps = True
                    move = self._random_non_losing_move(board, rng)
                    elapsed = time.perf_counter() - start
                else:
                    move = agent.best_move(board_for_agent)
                    elapsed = time.perf_counter() - start

                    if log_scores:
                        # Score logging is best-effort and should not affect legality enforcement.
                        try:
                            scores = agent.score_all_moves(board_for_agent)
                        except Exception as e:
                            logger.warning(
                                "score_all_moves() failed for agent=%s at ply=%d: %r",
                                agent_id, ply, e,
                            )
                            scores = None

            except Exception as e:
                # Agent exception => loss for that agent
                elapsed = time.perf_counter() - start
                winner = Color.RED if player == Color.YELLOW else Color.YELLOW
                logger.warning(
                    "Agent exception: agent=%s player=%s ply=%d err=%r",
                    agent_id, player.name, ply, e,
                )
                return GameRecord(
                    game_cfg=game_cfg,
                    moves=tuple(moves),
                    move_meta=tuple(meta),
                    winner=winner,
                    termination=TerminationReason.EXCEPTION,
                    detail=f"agent={agent_id} err={repr(e)}",
                )

            # Time accounting (best-effort)
            remaining_budget: float | None
            if player == Color.YELLOW:
                if budget_y is not None:
                    budget_y -= elapsed
                remaining_budget = budget_y
            else:
                if budget_r is not None:
                    budget_r -= elapsed
                remaining_budget = budget_r

            # Record meta for the decision (even if it ends game immediately after)
            meta.append(
                MoveMeta(
                    ply=ply,
                    player=player,
                    agent_id=agent_id,
                    epsilon=float(eps),
                    move=int(move),
                    was_epsilon_random=was_eps,
                    elapsed_s=float(elapsed),
                    remaining_budget_s=(None if remaining_budget is None else float(remaining_budget)),
                    scores=scores,
                )
            )

            # Timeout checks (best-effort: we can only decide now)
            if time_control.per_move_timeout_s is not None and elapsed > time_control.per_move_timeout_s:
                winner = Color.RED if player == Color.YELLOW else Color.YELLOW
                logger.warning(
                    "Per-move timeout: agent=%s player=%s ply=%d elapsed=%.6f limit=%.6f",
                    agent_id, player.name, ply, elapsed, time_control.per_move_timeout_s,
                )
                return GameRecord(
                    game_cfg=game_cfg,
                    moves=tuple(moves),
                    move_meta=tuple(meta),
                    winner=winner,
                    termination=TerminationReason.TIMEOUT,
                    detail=(
                        f"per-move timeout: agent={agent_id} elapsed={elapsed:.6f}s "
                        f"limit={time_control.per_move_timeout_s:.6f}s"
                    ),
                )

            if remaining_budget is not None and remaining_budget < 0.0:
                winner = Color.RED if player == Color.YELLOW else Color.YELLOW
                logger.warning(
                    "Per-game budget exceeded: agent=%s player=%s ply=%d remaining=%.6f",
                    agent_id, player.name, ply, remaining_budget,
                )
                return GameRecord(
                    game_cfg=game_cfg,
                    moves=tuple(moves),
                    move_meta=tuple(meta),
                    winner=winner,
                    termination=TerminationReason.TIMEOUT,
                    detail=f"per-game budget exceeded: agent={agent_id} remaining={remaining_budget:.6f}s",
                )

            # Illegal move => immediate loss
            if not isinstance(move, int) or not board.is_legal_move(int(move)):
                winner = Color.RED if player == Color.YELLOW else Color.YELLOW
                logger.warning(
                    "Illegal move: agent=%s player=%s ply=%d move=%r",
                    agent_id, player.name, ply, move,
                )
                return GameRecord(
                    game_cfg=game_cfg,
                    moves=tuple(moves),
                    move_meta=tuple(meta),
                    winner=winner,
                    termination=TerminationReason.ILLEGAL_MOVE,
                    detail=f"illegal move: agent={agent_id} move={repr(move)}",
                )

            # Apply move
            ok = board.play(int(move))
            if not ok:
                # Extremely defensive: `is_legal_move` said OK but play failed.
                winner = Color.RED if player == Color.YELLOW else Color.YELLOW
                logger.warning(
                    "Move application failed: agent=%s player=%s ply=%d move=%r",
                    agent_id, player.name, ply, move,
                )
                return GameRecord(
                    game_cfg=game_cfg,
                    moves=tuple(moves),
                    move_meta=tuple(meta),
                    winner=winner,
                    termination=TerminationReason.ILLEGAL_MOVE,
                    detail=f"play() failed despite is_legal_move: agent={agent_id} move={repr(move)}",
                )

            moves.append(int(move))

        # Normal termination
        winner_int = board.winner()
        if winner_int is None:
            return GameRecord(
                game_cfg=game_cfg,
                moves=tuple(moves),
                move_meta=tuple(meta),
                winner=None,
                termination=TerminationReason.DRAW,
                detail=None,
            )

        winner = Color(winner_int)
        return GameRecord(
            game_cfg=game_cfg,
            moves=tuple(moves),
            move_meta=tuple(meta),
            winner=winner,
            termination=TerminationReason.CONNECT4,
            detail=None,
        )

    @staticmethod
    def _random_non_losing_move(board: Board, rng: random.Random) -> int:
        candidates = board.legal_moves(non_losing=True)
        if candidates:
            return rng.choice(candidates)
        # Forced-loss state: pick any legal move
        all_legal = board.legal_moves()
        # Defensive: if empty, game should be over, but avoid crash
        return rng.choice(all_legal) if all_legal else 0


# -----------------------------
# Helpers
# -----------------------------

def _derive_game_seed(
    global_seed: int,
    agent_a: str,
    agent_b: str,
    # include assignment info so the two color-swapped schedules differ deterministically
    a_as: Color,
    b_as: Color,
    # epsilons are agent-owned (eps_a for agent_a, eps_b for agent_b)
    eps_a: float,
    eps_b: float,
    game_idx: int,
) -> int:
    """Derive a deterministic per-game seed from the global seed and matchup tuple.

    This makes randomness stable even if scheduling order changes.
    """
    payload = (
        f"{global_seed}|{agent_a}|{agent_b}|{int(a_as)}|{int(b_as)}|"
        f"{eps_a:.10f}|{eps_b:.10f}|{game_idx}"
    )
    digest = hashlib.blake2b(payload.encode("utf-8"), digest_size=8).digest()
    # Convert 8 bytes -> int in [0, 2^64)
    return int.from_bytes(digest, byteorder="little", signed=False)


def _aggregate_games(games: list[GameRecord]) -> list[AggregateRow]:
    """Aggregate per (agent_yellow, agent_red, eps_y, eps_r)."""
    # Key: (yellow_id, red_id, eps_y, eps_r)
    agg: dict[tuple[str, str, float, float], dict[str, int]] = {}

    for g in games:
        y_id = g.game_cfg.players.yellow_id
        r_id = g.game_cfg.players.red_id
        eps_y = g.game_cfg.epsilon_by_agent[y_id]
        eps_r = g.game_cfg.epsilon_by_agent[r_id]

        key = (y_id, r_id, eps_y, eps_r)

        if key not in agg:
            agg[key] = dict(
                games=0,
                yellow_wins=0,
                red_wins=0,
                draws=0,
                timeouts=0,
                illegal_moves=0,
                exceptions=0,
            )

        a = agg[key]
        a["games"] += 1

        if g.termination == TerminationReason.DRAW:
            a["draws"] += 1
        elif g.winner == Color.YELLOW:
            a["yellow_wins"] += 1
        elif g.winner == Color.RED:
            a["red_wins"] += 1

        if g.termination == TerminationReason.TIMEOUT:
            a["timeouts"] += 1
        elif g.termination == TerminationReason.ILLEGAL_MOVE:
            a["illegal_moves"] += 1
        elif g.termination == TerminationReason.EXCEPTION:
            a["exceptions"] += 1

    out: list[AggregateRow] = []
    for (y_id, r_id, eps_y, eps_r), d in agg.items():
        out.append(
            AggregateRow(
                agent_yellow=y_id,
                agent_red=r_id,
                epsilon_yellow=eps_y,
                epsilon_red=eps_r,
                games=d["games"],
                yellow_wins=d["yellow_wins"],
                red_wins=d["red_wins"],
                draws=d["draws"],
                timeouts=d["timeouts"],
                illegal_moves=d["illegal_moves"],
                exceptions=d["exceptions"],
            )
        )

    # Stable ordering for convenience
    out.sort(key=lambda row: (row.agent_yellow, row.agent_red, row.epsilon_yellow, row.epsilon_red))
    return out

In [ ]:
# -----------------------------
# 2) Arena config
# -----------------------------

# Logger is optional; arena is silent by default unless you configure logging.
logger = logging.getLogger("bitbully.arena")
logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout
# If you want to see logs in console:
# logging.basicConfig(level=logging.WARNING)

cfg = ArenaConfig(
    agents=(
        AgentSpec(
            agent_id="bitbully_12plydist_random",
            agent=bitbully_agent,
            colors=(Color.YELLOW, Color.RED),  # can play both
            epsilons=(0.00, 0.05, 0.10, 0.15, 0.2, 0.25),
        ),
        AgentSpec(
            agent_id="td_center",
            agent=td_agent,
            colors=(Color.YELLOW, Color.RED),  # can play both,
            epsilons=(0.00,),
        ),
    ),
    n_games=50,  # n games per pairing per ε per color-swap
    time_control=TimeControl(
        per_move_timeout_s=2.0,  # best-effort (measured after return)
        per_game_budget_s=45.0,    # seconds of total think time
    ),
    seed=12345,
    log_scores=False,   # set True to also call score_all_moves() for logging
    use_tqdm=True,      # optional progress bar
    logger=logger,
)

# -----------------------------
# 3) Run arena
# -----------------------------

arena = BitBullyArena()
result = arena.run(cfg)

In [26]:
# -----------------------------
# 4) Inspect results
# -----------------------------

print("Skipped pairings:", len(result.skipped))
print("Games played:", len(result.games))
print("Aggregate rows:", len(result.aggregates))

# Print aggregate rows (one row per (yellow_agent, red_agent, eps_y, eps_r))
for row in result.aggregates:
    print(row)

# Look at a single game record (includes move list + per-move metadata)
g0 = result.games[0]
print("First game:", g0.game_cfg)
print("Termination:", g0.termination, "winner:", g0.winner)
print("Moves:", g0.moves)
print("First move meta:", g0.move_meta[0])

Skipped pairings: 0
Games played: 600
Aggregate rows: 12
AggregateRow(agent_yellow='bitbully_12plydist_random', agent_red='td_center', epsilon_yellow=0.0, epsilon_red=0.0, games=50, yellow_wins=50, red_wins=0, draws=0, timeouts=0, illegal_moves=0, exceptions=0)
AggregateRow(agent_yellow='bitbully_12plydist_random', agent_red='td_center', epsilon_yellow=0.05, epsilon_red=0.0, games=50, yellow_wins=42, red_wins=8, draws=0, timeouts=0, illegal_moves=0, exceptions=0)
AggregateRow(agent_yellow='bitbully_12plydist_random', agent_red='td_center', epsilon_yellow=0.1, epsilon_red=0.0, games=50, yellow_wins=32, red_wins=12, draws=6, timeouts=0, illegal_moves=0, exceptions=0)
AggregateRow(agent_yellow='bitbully_12plydist_random', agent_red='td_center', epsilon_yellow=0.15, epsilon_red=0.0, games=50, yellow_wins=27, red_wins=18, draws=5, timeouts=0, illegal_moves=0, exceptions=0)
AggregateRow(agent_yellow='bitbully_12plydist_random', agent_red='td_center', epsilon_yellow=0.2, epsilon_red=0.0, game